# PYXIS Event Analysis — linear pandas walkthrough

This notebook rebuilds the workbook’s **Medication Consistency Analysis** as a sequence of transparent pandas transformations. It starts from the transaction export, reconstructs the device–medication rollup, and then calculates hospital-wide consistency, dominant unit, time trend, utilization tier, and stocking recommendation.

**Source workbook:** `PYXIS_EventAnalysis.xlsx` (same folder as this notebook by default). Update `WORKBOOK_PATH` if it is elsewhere. The workbook’s Settings tab supplies the analysis thresholds; event-level source rows are read from **All Device Event Report** and medication classifications from **Med Map**.

> The workbook’s “Early Period Volume” is defined as `max(0, total volume − recent volume)`. It is not a fixed historical window. The notebook retains that definition and labels the limitation. “Recent” means the latest 30 days, matching the workbook setting.

## 1. Imports and workbook location

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

WORKBOOK_PATH = Path("PYXIS_EventAnalysis.xlsx")
if not WORKBOOK_PATH.exists():
    # Common location when the notebook is opened from the project root.
    WORKBOOK_PATH = Path("/Users/jts/Downloads/PYXIS_EventAnalysis.xlsx")

assert WORKBOOK_PATH.exists(), f"Workbook not found: {WORKBOOK_PATH}"
WORKBOOK_PATH

## 2. Load settings and source tables

In [ ]:
settings_raw = pd.read_excel(WORKBOOK_PATH, sheet_name="Settings", usecols="A:B")
settings_raw = settings_raw.dropna(subset=["Setting"]).set_index("Setting")["Value"]

# The settings table uses labels as keys so workbook row references are explicit here.
quick_reload_hours = float(settings_raw["Quick-reload window (hrs)"])
as_of_date = pd.to_datetime(settings_raw["As-of date"], errors="coerce")
recent_days = int(settings_raw["Recent-use lookback (days)"])
overstock_days = float(settings_raw["Overstock threshold (days on hand)"])
wide_stock_devices = int(settings_raw["Wide-stock threshold (devices)"])
immediate_reload_hours = float(settings_raw["Immediate reload exclusion (hrs)"])

events = pd.read_excel(WORKBOOK_PATH, sheet_name="All Device Event Report")
med_map = pd.read_excel(WORKBOOK_PATH, sheet_name="Med Map")

print(f"Loaded {len(events):,} transaction rows and {len(med_map):,} medication mapping rows")
print(f"Analysis date: {as_of_date:%Y-%m-%d}; recent-use window: {recent_days} days")
print(f"Quick reload window: {quick_reload_hours:g} hours; immediate pocket-move window: {immediate_reload_hours:g} hour(s)")

## 3. Normalize source fields and check key columns

In [ ]:
# Only fields used downstream are retained. Names follow the workbook's source headings.
required_event_columns = [
    "User ID", "Device", "Med ID", "Med Description", "Raw Tx Type",
    "Tx Type", "Transaction Date/Time", "Quantity", "Beg", "End",
    "Quantity Dispensed Since Previous", "Is Churn Reload", "Last Load Timestamp",
    "Stay Days At Checkout", "ingredient", "atcL3",
]
missing = sorted(set(required_event_columns) - set(events.columns))
if missing:
    raise ValueError(f"Source sheet is missing expected columns: {missing}")

events = events[required_event_columns].copy()
events["Transaction Date/Time"] = pd.to_datetime(events["Transaction Date/Time"], errors="coerce")
events["Last Load Timestamp"] = pd.to_datetime(events["Last Load Timestamp"], errors="coerce")
for col in ["Med ID", "Quantity", "Beg", "End", "Quantity Dispensed Since Previous", "Is Churn Reload", "Stay Days At Checkout"]:
    events[col] = pd.to_numeric(events[col], errors="coerce")

events["Device"] = events["Device"].astype("string").str.strip()
events["Tx Type"] = events["Tx Type"].astype("string").str.strip()
events["Med ID"] = events["Med ID"].astype("Int64")
events["device_med_key"] = events["Device"] + "|" + events["Med ID"].astype("string")

quality_checks = pd.Series({
    "rows": len(events),
    "missing transaction timestamps": events["Transaction Date/Time"].isna().sum(),
    "missing device IDs": events["Device"].isna().sum(),
    "missing medication IDs": events["Med ID"].isna().sum(),
    "duplicate source rows": events.duplicated().sum(),
})
quality_checks

## 4. Rebuild workbook event measures needed for the rollup

The workbook precomputes event-level columns in the source tab. This step reconstructs the fields needed by the summaries from the raw transaction columns: transaction type with immediate unload/reload moves excluded, positive dispensed quantities, quick reload flags, time since latest load, current quantity, and recent-use volume. The source export is expected to be ordered by device, medication, and event time; the code explicitly sorts before calculating sequences.

In [ ]:
events = events.sort_values(["Device", "Med ID", "Transaction Date/Time"], kind="stable").reset_index(drop=True)
next_type = events.groupby(["Device", "Med ID"])["Raw Tx Type"].shift(-1)
next_time = events.groupby(["Device", "Med ID"])["Transaction Date/Time"].shift(-1)
hours_until_next = (next_time - events["Transaction Date/Time"]).dt.total_seconds() / 3600

is_immediate_move = (
    events["Raw Tx Type"].eq("Unload")
    & next_type.notna()
    & next_type.ne("Unload")
    & hours_until_next.le(immediate_reload_hours)
)
events["analysis_tx_type"] = events["Raw Tx Type"].mask(is_immediate_move, "Pocket Move")

events["dispensed_since_previous"] = pd.to_numeric(events["Quantity Dispensed Since Previous"], errors="coerce")
events["dispensed_positive"] = events["dispensed_since_previous"].where(events["dispensed_since_previous"] > 0, 0).fillna(0)
events["is_quick_reload"] = (
    events["analysis_tx_type"].eq("Unload")
    & next_type.notna()
    & next_type.ne("Unload")
    & hours_until_next.le(quick_reload_hours)
).astype("int8")

if pd.isna(as_of_date):
    as_of_date = events["Transaction Date/Time"].max()
recent_start = as_of_date - pd.Timedelta(days=recent_days)
events["is_recent"] = events["Transaction Date/Time"].between(recent_start, as_of_date, inclusive="both")

print(f"Prepared events from {events['Transaction Date/Time'].min():%Y-%m-%d} to {events['Transaction Date/Time'].max():%Y-%m-%d}")
print("Raw transaction types:", events["Raw Tx Type"].value_counts(dropna=False).to_dict())
print("Analysis transaction types:", events["analysis_tx_type"].value_counts(dropna=False).to_dict())

## 5. Aggregate transactions to device–medication pairs

This reproduces the **Drug-Device Summary** grain: one row per device and medication. Workbook formulas use the first-to-last transaction span for average daily use, the latest transaction for current on-hand quantity, and the latest 30 days for recent use. A last transaction of `Unload` implies zero current quantity.

In [ ]:
events["is_unload"] = events["analysis_tx_type"].eq("Unload")

group_cols = ["Device", "Med ID"]
rollup = events.groupby(group_cols, dropna=False, sort=False).agg(
    Description=("Med Description", "first"),
    first_tx=("Transaction Date/Time", "min"),
    last_tx=("Transaction Date/Time", "max"),
    transaction_rows=("Transaction Date/Time", "size"),
    unloads=("is_unload", "sum"),
    quick_reloads=("is_quick_reload", "sum"),
    net_dispensed=("dispensed_positive", "sum"),
    recent_dispensed=("dispensed_positive", lambda s: s[events.loc[s.index, "is_recent"]].sum()),
    mean_loaded_days_before_unload=("Stay Days At Checkout", "mean"),
).reset_index()

last_events = events.sort_values("Transaction Date/Time").groupby(group_cols, dropna=False, sort=False).tail(1)
last_state = last_events[group_cols + ["analysis_tx_type", "End"]].rename(
    columns={"analysis_tx_type": "last_tx_type", "End": "last_end_quantity"}
)
rollup = rollup.merge(last_state, on=group_cols, how="left", validate="one_to_one")
rollup["days_observed"] = (rollup["last_tx"] - rollup["first_tx"]).dt.total_seconds() / 86400
rollup["avg_daily_use"] = rollup["net_dispensed"] / rollup["days_observed"].clip(lower=1)
rollup["current_qty"] = rollup["last_end_quantity"].where(rollup["last_tx_type"].ne("Unload"), 0).fillna(0)
rollup["quick_reload_rate"] = rollup["quick_reloads"] / rollup["unloads"].replace(0, np.nan)
rollup["days_on_hand_recent_use"] = np.where(
    rollup["current_qty"].eq(0), np.nan,
    np.where(rollup["recent_dispensed"].eq(0), np.inf,
             rollup["current_qty"] / (rollup["recent_dispensed"] / recent_days))
)
rollup["devices_stocking_med"] = rollup.groupby("Med ID")["Device"].transform("nunique")
rollup.head()

## 6. Apply device–medication recommendations

In [ ]:
def device_recommendation(row):
    if row["quick_reloads"] >= 2:
        return "Stop unloading – keep stocked"
    if row["current_qty"] > 0 and row["recent_dispensed"] == 0 and row["devices_stocking_med"] >= wide_stock_devices:
        return "Low use here but widely stocked – confirm before unloading"
    if row["current_qty"] > 0 and row["recent_dispensed"] == 0:
        return "Unload candidate – no recent use"
    if pd.notna(row["days_on_hand_recent_use"]) and row["days_on_hand_recent_use"] > overstock_days:
        return "Reduce par – overstocked"
    if row["quick_reloads"] == 1:
        return "Watch – 1 quick reload"
    return "OK"

rollup["device_recommendation"] = rollup.apply(device_recommendation, axis=1)
rollup[["Device", "Med ID", "transaction_rows", "unloads", "quick_reloads", "recent_dispensed", "current_qty", "days_on_hand_recent_use", "device_recommendation"]].head(10)

## 7. Join medication classification

In [ ]:
med_map = med_map.copy()
med_map["Med ID"] = pd.to_numeric(med_map["Med ID"], errors="coerce").astype("Int64")
map_cols = [c for c in ["Med ID", "Description", "Ingredient", "ATC level 3", "ATC class name"] if c in med_map.columns]
med_map_unique = med_map[map_cols].drop_duplicates("Med ID", keep="first")

class_events = rollup.merge(med_map_unique, on="Med ID", how="left", suffixes=("_event", "_map"), validate="many_to_one")
class_events["med_description"] = class_events["Description_map"].fillna(class_events["Description_event"])
class_events["atc_class"] = class_events["ATC class name"].fillna("Unclassified")
class_events[["Med ID", "med_description", "atc_class"]].head()

## 8. Consolidate to one hospital-wide row per medication

Metrics below match the workbook’s **Medication Consistency Analysis**: total net dispensed; device count; highest-volume unit (first device wins ties); primary unit share; sample standard deviation of unit-level volumes divided by their mean (CV); early volume as total minus recent volume; and trend `((recent × 2) − early) / early`. If early and recent are both zero trend is zero; if early is zero but recent is positive it is 100%.

In [ ]:
def summarize_medication(group):
    by_device = group.sort_values(["net_dispensed", "Device"], ascending=[False, True], kind="stable")
    total = by_device["net_dispensed"].sum()
    stocking_units = by_device["Device"].nunique()
    primary = by_device.iloc[0]["Device"] if total > 0 else "Multiple / Zero Use"
    mean_device = by_device["net_dispensed"].mean()
    cv = (by_device["net_dispensed"].std(ddof=1) / mean_device) if stocking_units > 1 and mean_device else 0.0
    recent = by_device["recent_dispensed"].sum()
    early = max(0.0, total - recent)
    trend = 0.0 if early == 0 and recent == 0 else (1.0 if early == 0 else ((recent * 2) - early) / early)
    primary_share = (by_device["net_dispensed"].max() / total) if total else 0.0
    return pd.Series({
        "Description": by_device["med_description"].iloc[0],
        "ATC Class": by_device["atc_class"].iloc[0],
        "Total Net Dispensed": total,
        "Stocking Units Count": stocking_units,
        "Primary Clinical Unit": primary,
        "Primary Unit Volume %": primary_share,
        "Cross-Unit Consistency (CV)": cv,
        "Early Period Volume": early,
        "Recent Period Volume": recent,
        "Temporal Trend %": trend,
    })

medication_summary = (
    class_events.groupby("Med ID", dropna=False, sort=True)
    .apply(summarize_medication, include_groups=False)
    .reset_index()
)
medication_summary.head()

## 9. Assign utilization stability tiers and stocking recommendations

In [ ]:
def stability_tier(row):
    if row["Total Net Dispensed"] == 0:
        return "Dormant / Zero Use"
    cv = row["Cross-Unit Consistency (CV)"]
    if cv < 0.6:
        return "Uniform Hospital-Wide"
    if cv <= 1.2:
        return "Moderate Variation"
    return "Erratic / Unit-Specific"

def stocking_recommendation(row):
    if row["Total Net Dispensed"] == 0:
        return "De-stock Candidate"
    if row["Stocking Units Count"] >= 10 and row["Total Net Dispensed"] <= 15:
        return "Consolidate Units"
    if row["Temporal Trend %"] < -0.4 and row["Recent Period Volume"] <= 5:
        return "Trending Down - Reduce Par"
    if row["Temporal Trend %"] > 0.5:
        return "Trending Up - Monitor Par"
    if row["Cross-Unit Consistency (CV)"] > 1.5:
        return "Concentrate in Primary Unit"
    return "Maintain Par"

medication_summary["Utilization Stability Tier"] = medication_summary.apply(stability_tier, axis=1)
medication_summary["Stocking Recommendation"] = medication_summary.apply(stocking_recommendation, axis=1)
medication_summary = medication_summary.sort_values("Med ID").reset_index(drop=True)
medication_summary.head(12)

## 10. Review headline metrics and highest-priority rows

In [ ]:
headline = pd.Series({
    "medications analyzed": medication_summary["Med ID"].nunique(),
    "total hospital dispense volume": medication_summary["Total Net Dispensed"].sum(),
    "average cabinet footprint": medication_summary["Stocking Units Count"].mean(),
    "high-volatility medications (CV > 1.0)": medication_summary["Cross-Unit Consistency (CV)"].gt(1.0).sum(),
    "recommendations to consolidate units": medication_summary["Stocking Recommendation"].eq("Consolidate Units").sum(),
    "recommendations to concentrate in primary unit": medication_summary["Stocking Recommendation"].eq("Concentrate in Primary Unit").sum(),
})
headline

In [ ]:
priority_view = medication_summary.sort_values(
    ["Stocking Recommendation", "Cross-Unit Consistency (CV)", "Total Net Dispensed"],
    ascending=[True, False, False]
)[["Med ID", "Description", "ATC Class", "Total Net Dispensed", "Stocking Units Count", "Primary Clinical Unit", "Cross-Unit Consistency (CV)", "Temporal Trend %", "Utilization Stability Tier", "Stocking Recommendation"]]
priority_view.head(25)

## 11. Optional exports

Run this cell to save the reconstructed hospital-wide analysis and device–medication detail as CSV files in `outputs/`.

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
medication_summary.to_csv(OUTPUT_DIR / "pyxis_medication_consistency.csv", index=False)
class_events.to_csv(OUTPUT_DIR / "pyxis_device_medication_rollup.csv", index=False)
print("Saved:", OUTPUT_DIR / "pyxis_medication_consistency.csv")
print("Saved:", OUTPUT_DIR / "pyxis_device_medication_rollup.csv")

## Interpretation notes

1. Results inherit the source workbook’s transaction semantics and configured 30-day recent-use lookback. Recommendations are inventory planning signals for review, not automated clinical decisions.
2. Current quantity uses the latest event balance (`End`) except when the latest event is an unload, which is treated as zero.
3. The workbook’s trend baseline is residual historical volume rather than a matching-length prior period. For a true period-over-period rate, define a separate comparison window and recalculate early and recent dispense volumes from event dates.
4. Medication mapping may have missing or duplicate source IDs. The join keeps the first mapping per medication ID; review `med_map[med_map["Med ID"].duplicated(keep=False)]` if mapping conflicts matter.